# مدير Google Drive عبر Telegram

هذا الدفتر يجهز المشروع داخل Google Colab، ويربط Google Drive عبر واجهة Colab التفاعلية الرسمية. يتطلب تشغيل Telegram إدخال Bot Token وAPI ID وAPI Hash ومعرف المستخدم الإداري. لا تضع الأسرار في خلايا ثابتة ولا تشارك الدفتر بعد تشغيل خلايا الأسرار.

حمّل ملف ZIP للمشروع إلى جهازك قبل فتح هذا الدفتر؛ إذا لم يكن مجلد المشروع موجودًا في MyDrive، ستظهر نافذة رفع لاختياره واستخراجه.

In [ ]:
from google.colab import drive

# تظهر شاشة Google الرسمية للموافقة على ربط Drive.
drive.mount("/content/drive")

In [ ]:
import sys
import zipfile
from pathlib import Path

from google.colab import files

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "drive-telegram-manager"
if not (PROJECT_DIR / "pyproject.toml").is_file():
    print("ارفع أرشيف drive-telegram-manager.zip الذي نزّلته من التسليم.")
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if not archives:
        raise RuntimeError("لم يتم اختيار ملف ZIP للمشروع.")
    archive = archives[0]
    with zipfile.ZipFile(archive) as zf:
        destination = DRIVE_ROOT.resolve()
        for member in zf.infolist():
            resolved = (DRIVE_ROOT / member.filename).resolve()
            if resolved != destination and destination not in resolved.parents:
                raise ValueError(f"مسار غير آمن داخل الأرشيف: {member.filename}")
        zf.extractall(DRIVE_ROOT)
    archive.unlink(missing_ok=True)
if not (PROJECT_DIR / "pyproject.toml").is_file():
    raise FileNotFoundError(f"لم أعثر على ملفات المشروع داخل {PROJECT_DIR}")
bundled_api = PROJECT_DIR / ".local" / "bin" / "telegram-bot-api"
if bundled_api.is_file():
    bundled_api.chmod(0o755)  # استعادة صلاحية التنفيذ التي قد لا يحفظها zipfile
sys.path.insert(0, str(PROJECT_DIR / "src"))
print("مجلد المشروع:", PROJECT_DIR)

In [ ]:
import os
import platform
import shutil
import subprocess

print("Python:", platform.python_version(), "| Arch:", platform.machine())
assert shutil.which("bash"), "bash غير متوفر في runtime"
if not shutil.which("uv"):
    subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, check=True)
    os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ.get("PATH", "")
print(subprocess.check_output(["uv", "--version"], text=True).strip())

In [ ]:
import subprocess

# التثبيت قابل لإعادة التنفيذ، ويستخدم uv.lock واعتماديات المشروع.
subprocess.run(["bash", str(PROJECT_DIR / "install-colab.sh")], cwd=PROJECT_DIR, check=True)

In [ ]:
import getpass
import os

# تبقى القيم في بيئة runtime فقط ولا تُطبع. احذفها عند الانتهاء من العمل.
os.environ["BOT_TOKEN"] = getpass.getpass("BOT_TOKEN من BotFather: ")
os.environ["TELEGRAM_API_ID"] = getpass.getpass("TELEGRAM_API_ID من my.telegram.org: ")
os.environ["TELEGRAM_API_HASH"] = getpass.getpass("TELEGRAM_API_HASH من my.telegram.org: ")
os.environ["ADMIN_IDS"] = getpass.getpass("Telegram user ID المصرح به: ")
os.environ.setdefault("TELEGRAM_LOCAL_API_URL", "http://127.0.0.1:8081")
os.environ.setdefault("DRIVE_MOUNT_POINT", "/content/drive")
os.environ.setdefault("DRIVE_ROOT", "/content/drive/MyDrive")
os.environ.setdefault("TEMP_DIR", "/content/dtm-tmp")
print("تم إدخال الإعدادات السرية دون عرضها.")

In [ ]:
drive_root = Path(os.environ["DRIVE_ROOT"])
if not drive_root.is_dir():
    raise FileNotFoundError(f"جذر Drive غير متاح: {drive_root}. أعد تشغيل drive.mount أولًا.")
print("Google Drive متصل:", drive_root)
print("مساحة Colab الحرة:", f"{shutil.disk_usage('/content').free:,} bytes")

## المصادقة الإضافية في Colab

الوظائف الأساسية للملفات تستخدم mount ولا تحتاج Google API Key أو Service Account. مصادقة Colab OAuth الإضافية متاحة فقط لعمليات Drive API الوصفية التي تتطلبها؛ هذه النسخة الحالية لا تعرض أوامر تغيير صلاحيات المشاركة. لا تنفذ المصادقة إلا إذا كانت الوظيفة الإضافية التي تستخدمها تحتاجها.

In [ ]:
# اختياري: مصادقة Colab التفاعلية لخدمات Google API التي تحتاجها وحدات إضافية.
# لا تُنشئ API key ولا ترفع credentials.json.
ENABLE_COLAB_DRIVE_API_AUTH = False
if ENABLE_COLAB_DRIVE_API_AUTH:
    from drive_telegram_manager.drive.mount import authenticate_drive_api

    drive_service = authenticate_drive_api()
    print("اكتملت مصادقة Colab API.")

In [ ]:
# تثبيت الحزمة المحلية من المشروع، ثم فحص الاتصال بالخادم بعد إعداد الأسرار.
subprocess.run(["uv", "sync", "--frozen", "--all-extras"], cwd=PROJECT_DIR, check=True)
subprocess.run(["bash", str(PROJECT_DIR / "healthcheck.sh")], cwd=PROJECT_DIR, check=True)

In [ ]:
# يبدأ run.sh الخادم والبوت في الخلفية مع تسجيل PID؛ لا ينشئ نسخة ثانية عند إعادة التنفيذ.
subprocess.run(["bash", str(PROJECT_DIR / "run.sh")], cwd=PROJECT_DIR, check=True)
print("سجل البوت محفوظ في:", PROJECT_DIR / "state" / "bot.log")

In [ ]:
# نفّذ السطر المناسب عند الحاجة؛ healthcheck لا يعرض قيم الأسرار.
subprocess.run(["bash", str(PROJECT_DIR / "healthcheck.sh")], cwd=PROJECT_DIR, check=False)
# للإيقاف: subprocess.run(['bash', str(PROJECT_DIR / 'stop.sh')], cwd=PROJECT_DIR, check=True)
# لإعادة التشغيل: subprocess.run(['bash', str(PROJECT_DIR / 'restart.sh')], cwd=PROJECT_DIR, check=True)
print("لا تعرض محتوى state/bot.log أو bot-api.log علنًا قبل مراجعة الأسرار.")

## تنبيهات تشغيل

جلسة Colab مؤقتة. قد تختفي الملفات المحلية تحت `/content` وتنقطع العمليات. لا يغني Drive mount عن وجود مساحة محلية كافية للملفات المرحلية أو لبناء Local Bot API Server. إذا انتهت الجلسة، أعد mount وأعد إدخال الأسرار، ثم تحقق من سجل الوظائف والملفات في الوجهة قبل إعادة أي عملية. لا يعيد البوت عمليات النقل المنقطعة تلقائيًا. أوقف الخادم والبوت عبر `stop.sh` عندما تكون الجلسة مستمرة.